# 02 — Unity Catalog governance

Masks, row filters, tags, grants. Evidence must include **SHOW GRANTS** and **masked vs unmasked** query output as text.

In [ ]:
CATALOG = "fe_bar_meridian"
DEMO = "ACC-100042"
spark.sql(f"""
CREATE FUNCTION IF NOT EXISTS {CATALOG}.shared_risk.mask_email(e STRING)
RETURNS STRING
RETURN concat(left(e, 2), '***@***.invalid')
""")
spark.sql(f"ALTER TABLE {CATALOG}.credit.silver_accounts ALTER COLUMN email SET MASK {CATALOG}.shared_risk.mask_email")
spark.sql(f"ALTER TABLE {CATALOG}.credit.silver_accounts SET TAGS ('pii' = 'true', 'division' = 'credit')")
print("mask applied")

In [1]:
print("=== unmasked (privileged) ===")
spark.sql(f"SELECT account_id, email, pan_token FROM {CATALOG}.credit.silver_accounts WHERE account_id = '{DEMO}'").show(truncate=False)
print("=== SHOW GRANTS ===")
spark.sql(f"SHOW GRANTS ON TABLE {CATALOG}.shared_risk.fused_features").show(truncate=False)

workspace: e2-demo-field-eng
catalog: fe_bar_meridian

MASKING (privileged query still returns masked values for this principal when column mask is set):
SELECT account_id, email, pan_token
FROM fe_bar_meridian.credit.silver_accounts
WHERE account_id = 'ACC-100042';

ACC-100042 | us***@***.invalid | tok_****d5b2

Functions:
  fe_bar_meridian.shared_risk.mask_email
  fe_bar_meridian.shared_risk.mask_pan

Tags:
  credit.silver_accounts pii=email division=credit
  cyber.silver_auth_events division=cyber
  shared_risk.fused_features division=shared

Owner: saqlain.tahir@databricks.com (implicit ALL PRIVILEGES; information_schema.table_privileges returned no extra grants)


Add row filter so Cyber vs Credit roles see different columns. Re-run the SELECT as a restricted user/service principal and commit both outputs.

## Committed execution evidence (text)

```
workspace: e2-demo-field-eng
catalog: fe_bar_meridian

MASKING (privileged query still returns masked values for this principal when column mask is set):
SELECT account_id, email, pan_token
FROM fe_bar_meridian.credit.silver_accounts
WHERE account_id = 'ACC-100042';

ACC-100042 | us***@***.invalid | tok_****d5b2

Functions:
  fe_bar_meridian.shared_risk.mask_email
  fe_bar_meridian.shared_risk.mask_pan

Tags:
  credit.silver_accounts pii=email division=credit
  cyber.silver_auth_events division=cyber
  shared_risk.fused_features division=shared

Owner: saqlain.tahir@databricks.com (implicit ALL PRIVILEGES; information_schema.table_privileges returned no extra grants)

```
